# Shortest Paths and Travel Time

**Series:** Network Analysis (notebook 2 of 3)
**Time:** about 25 minutes

**In this notebook you will:**
- Snap a coordinate to the nearest intersection
- Find the shortest route by distance and by travel time
- Measure a route's length in kilometers and minutes
- Color every intersection by its travel time from one point

**You should already know:** notebook 02a (graphs, nodes and edges).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## Step 1: Install, import and load the network

In [ ]:
!pip install -q osmnx

import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
place = "West Lafayette, Indiana, USA"
G = ox.graph_from_place(place, network_type="drive")

## Step 2: Add speeds and travel times

OSM stores speed limits for many roads. `add_edge_speeds` fills in missing ones using typical
speeds for that road type. If a road type has no speed data at all, `fallback` (in km/h) is used.
`add_edge_travel_times` then divides length by speed to get seconds.

In [ ]:
G = ox.add_edge_speeds(G, fallback=40)   # 40 km/h for road types with no speed data
G = ox.add_edge_travel_times(G)

## Step 3: Snap two locations to the network

Routes run between nodes, so we find the node closest to each location.
**Careful:** `nearest_nodes` takes **longitude first** (`X`), then latitude (`Y`).

In [ ]:
# Purdue Memorial Union and a shopping area on the north side of town
start_lat, start_lon = 40.4247, -86.9112
end_lat, end_lon = 40.4580, -86.9290

start_node = ox.nearest_nodes(G, X=start_lon, Y=start_lat)
end_node = ox.nearest_nodes(G, X=end_lon, Y=end_lat)
print(start_node, end_node)

## Step 4: Find the shortest route

`weight` decides what "shortest" means: `"length"` for distance or `"travel_time"` for time.

In [ ]:
route_short = ox.shortest_path(G, start_node, end_node, weight="length")
route_fast = ox.shortest_path(G, start_node, end_node, weight="travel_time")

fig, ax = ox.plot_graph_route(G, route_short, route_color="red", node_size=0, bgcolor="white")

## Step 5: How long is the route?

`route_to_gdf` returns the route's edges as a table, so we can add up their lengths and times.

In [ ]:
route_edges = ox.routing.route_to_gdf(G, route_fast)

km = route_edges["length"].sum() / 1000
minutes = route_edges["travel_time"].sum() / 60
print(f"Fastest route: {km:.2f} km, about {minutes:.1f} minutes")

## Step 6: See the route on an interactive map

In [ ]:
route_edges.explore(color="red", style_kwds={"weight": 5}, tiles="Esri.WorldGrayCanvas")

## Step 7: Travel time from one point to everywhere

Dijkstra's algorithm finds the fastest time from one node to **every** other node.
This is the core idea behind access maps.

In [ ]:
times = nx.single_source_dijkstra_path_length(G, start_node, weight="travel_time")

nodes, edges = ox.graph_to_gdfs(G)
nodes["minutes"] = nodes.index.map(times) / 60

ax = edges.plot(figsize=(8, 8), linewidth=0.4, color="lightgray")
nodes.plot(ax=ax, column="minutes", cmap="viridis", markersize=8, legend=True,
           legend_kwds={"label": "Minutes from start"})
ax.set_title("Drive time from Purdue Memorial Union")
plt.show()

Intersections close to the start are dark purple; those farther away turn yellow.
Gray background lines are the roads.

## Try it yourself

Compare the two routes from Step 4. Which is longer in kilometers, `route_short` or `route_fast`? Which is quicker in minutes?

*Hint:* reuse the code from Step 5 for each route.

<details>
<summary><b>Show a solution</b></summary>

```python
for name, r in [("shortest", route_short), ("fastest", route_fast)]:
    e = ox.routing.route_to_gdf(G, r)
    print(name, round(e["length"].sum() / 1000, 2), "km,",
          round(e["travel_time"].sum() / 60, 1), "min")
```

</details>

In [ ]:
# Write your answer here

## Recap

- `nearest_nodes` snaps a coordinate to the network (longitude first).
- `shortest_path` finds the best route by `length` or `travel_time`.
- `route_to_gdf` lets you add up a route's distance and time.
- Dijkstra's algorithm gives travel time from one point to every intersection.

**Next:** [Travel Time to Grocery Stores](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/02c_Travel_Time_to_Stores.ipynb)